Goal: Rebuild legacy R code to get the WMC dashboard metrics using the .csv files generated from main

In [21]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

In [22]:
# Will try to keep this the same for easier automation in the future
year = "25"
month = "10"

# set working directories - can change in future, be careful
root_dir = "K:/AP/TTM/"
busstate_dir = os.path.join(root_dir, "Data/WMC Dashboard/BusState Cleaned") # Where cleaned busstate data is stored
stops_dir = os.path.join(root_dir, "Data/WMC Dashboard/stops") # where a copy of previous stops data is stored
# NOTE: stops data WILL need to be updated with new med center stops

# pull in necessary static files NOTE: again these WILL need to be updated with new med center stops - may break things downstream so be careful after changing
stop_inventory = pd.read_csv(os.path.join(stops_dir, "stop_inventory.csv"))
pattern_stops = pd.read_csv(os.path.join(stops_dir, "pattern_stops.csv"), header=None)

# have to clean up year/month for proper file reading
# simple map for month number to month abbreviation for file reading
months = {"01": "JAN","02": "FEB","03": "MAR","04": "APR","05": "MAY","06": "JUN","07": "JUL","08": "AUG","09": "SEP","10": "OCT","11": "NOV","12": "DEC"}

year_full = 2000 + int(year)
month_full = months[month]

# pull in cleaned busstate data for month and year of interest
busstate = pd.read_csv(os.path.normpath(os.path.join(busstate_dir, f"{year_full}-{month_full}-busstate.csv")))

In [23]:
# set directory to save dashbaord data
dashboard_dir = os.path.join(root_dir, f"Data/WMC Dashboard/Dashboard Data/{year_full}/{month_full}")
Path(dashboard_dir).mkdir(parents=True, exist_ok=True)

In [24]:
# original R code selected the 1st and 10th cols
pattern_stops_filtered = pattern_stops.iloc[:, [0, 9]].drop_duplicates()
# name schema per original R code
pattern_stops_filtered.columns = ["ROUTE", "STOP_ID"]

# left join stop inventory 
stops = pattern_stops_filtered.merge(stop_inventory, on="STOP_ID", how="left")

In [ ]:
# stops.info()
# sum(stops['ROUTE'] == "MC")

# print(stops[stops['ROUTE'] == "MC"])

<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ROUTE           73 non-null     str    
 1   STOP_ID         73 non-null     int64  
 2   STOP_NAME       72 non-null     str    
 3   TIMEPOINT_NAME  38 non-null     str    
 4   LONG            72 non-null     float64
 5   LAT             72 non-null     float64
 6   HEADING         72 non-null     float64
dtypes: float64(3), int64(1), str(3)
memory usage: 4.1 KB
   ROUTE  STOP_ID            STOP_NAME TIMEPOINT_NAME       LONG        LAT  \
59    MC      403            CARMACK 2          CMCK2 -83.037180  40.001046   
60    MC      404            CARMACK 3          CMCK3 -83.040177  40.000878   
61    MC      405    JOHN HERRICK LOOP           THUB -83.018119  39.997569   
62    MC       27  Outbound to Carmack       OUTBMC86 -83.021652  39.997636   
63    MC       94   CARMACK 5 + STOP 1            N

In [31]:
# create distance function
# NOTE: This does NOT account for curvature of the earth, but locations are close enough that it is negligible
# NOTE: If we want to be more precise in the future, this function can be updated.
def distance(x1, x2, y1, y2):
    '''
    Calculate the distance between two points.

    Args:
        x1 (float): The x-coordinate of the first point longitude.
        x2 (float): The x-coordinate of the second point longitude.
        y1 (float): The y-coordinate of the first point latitude.
        y2 (float): The y-coordinate of the second point latitude.
    Returns:
        float: The distance between the two points.
    '''
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

In [ ]:
# Create function to determine the closest stop
def which_stop(lat, long,  route = "MC", stops = stops):
    '''
    Determine the closest stop to a given latitude and longitude. Med center route is default, but can be updated to other routes as needed. 
    NOTE: This will potentially break when new stops are added start Dec. 2025 - be careful when updating stop inventory.

    Args:
        lat (float): The latitude of the point of interest.
        long (float): The longitude of the point of interest.
        route (str): The route to filter stops by. Default is "MC" for medical center. Based on ROUTE col in stops DataFrame.
        stops (DataFrame): A DataFrame containing stop information, including 'STOP_ID', 'LAT', and 'LONG' columns.

    Returns:
        int: The STOP_ID of the closest stop.
    '''
    # isolate the specified route stops
    selected_stops = stops[stops['ROUTE'] == route].copy()

    # Calculate distance to each stop
    selected_stops['DISTANCE'] = distance(long, selected_stops['LONG'], lat, selected_stops['LAT'])

    # subsetting only stops within 0.0005 units
    selected_stops = selected_stops[selected_stops['DISTANCE'] < 0.0005] # do not know exact units, but this was the threshold used in original R code
    selected_stops = selected_stops.sort_values('DISTANCE')

    # sorted by distance, therfore return first stop which is closest
    return selected_stops.iloc[0]["STOP_ID"]



In [33]:
which_stop(40.001045, -83.037181)

np.int64(403)